# AI4EAC Liquidity Stress Prediction: End-to-End Champion Pipeline

This notebook executes the complete 5-stage leak-free tournament reproduction and ensemble stacking pipeline.

### Pipeline Configuration:
- **Cross-Validation**: 1-Seed (seed=42), 10-Fold Stratified CV across all stages.
- **Validation Integrity**: Strictly nested feature screening, zero validation fold leakage, and genuine out-of-sample calibration.

### Stage Architecture:
1. **Stage 1**: Baseline Family Anchor (1-Seed 10-Fold CatBoost, XGBoost, LightGBM).
2. **Stage 2**: 1-Seed 10-Fold Domain GBDT Zoo (Nested feature screening + resilient fold metadata across 5 architectures).
3. **Stage 3**: Dual TabPFN Foundation Priors (Physics & Solvency ratio representations).
4. **Stage 4**: Diversity Base Learners (MultiStrata Stratified Ensemble + Teacher-Student Distillation + Neural TabMLP).
5. **Stage 5**: 10-Fold Cross-Validated Logit-Space L2 Regularized Meta-Stacker + Single-Stage Cross-Fitted Calibration.
6. **Stage 6**: Automated Competition Invariant Audit (30k rows, probability clipping, zero NaNs, natural prevalence).
7. **Stage 7**: 20% Immutable Shadow Holdout Verification Benchmark.

## Stage 1: Tournament Baseline Anchor Reproduction & Hill Climbing
Trains 1-seed (seed=42) 10-fold CatBoost, XGBoost, and HistGradientBoosting on the selected features, optimizes weights via SLSQP, applies calibration, and runs stepwise hill climbing.

In [ ]:
from src.stages.stage1_reproduce import run_stage1
import time
t0 = time.time()
s1_score = run_stage1()
print(f"Stage 1 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s1_score:.5f}")


## Stage 2: 10-Fold Multi-Model Domain GBDT Zoo
Executes CatBoost, XGBoost, and LightGBM with leak-free fold-nested feature screening across specialized domain subsets (solvency ratios, balance collapses, and triage columns).

In [ ]:
from src.stages.stage2_gbdt_zoo import run_stage2
import time
t0 = time.time()
s2_score = run_stage2(seeds=(42,), n_splits=10)
print(f"Stage 2 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s2_score:.5f}")


## Stage 3: Dual TabPFN Foundation Priors
Extracts non-tree foundation model representations on GPU using RankGauss Quantile Transformation on the physical solvency ratio views.

In [ ]:
import time
from src.stages.stage3_tabpfn_priors import run_stage3

t0 = time.time()
s3_score = run_stage3(n_splits=10)
print(f"Stage 3 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s3_score:.5f}")


## Stage 4: Diversity Pipeline (MultiStrata, Distillation Students, Neural TabMLP)
Executes:
1. MultiStrata 5-Fold Iterative Stratification Ensemble (906 features).
2. 10-Fold Teacher-Student Distillation (CatBoost & XGBoost) on clean soft probability targets.
3. PyTorch Neural TabMLP with residual connections and bounded logit scaling.

In [ ]:
import time
from src.stages.stage4_diversity import run_stage4

t0 = time.time()
s4_score = run_stage4()
print(f"Stage 4 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s4_score:.5f}")


## Stage 5: 10-Fold Cross-Validated Logit-Space L2 Meta-Stacker
Solves analytical L-BFGS-B regularized regression in logit space across all candidate model streams, eliminates in-sample hill-climbing, and applies single-stage cross-fitted Platt calibration.

In [ ]:
import time
from src.stages.stage5_meta_stacker import run_stage5

t0 = time.time()
final_champion_score = run_stage5()
print(f"Stage 5 complete: Champion Score = {final_champion_score:.5f}")


## Stage 6: Final Verification & Invariants Audit
Checks all 5 hard competition invariants: exact row counts (30,000), column names, zero NaNs, probability bounding [0.002, 0.998], and benchmark correlation.

In [ ]:
import os
import pandas as pd
from src.stages.audit import run_audit

# Run automated audit
audit_passed = run_audit()
print(f"Audit Passed: {audit_passed}")

# Inspect final submission preview
sub_file = "submissions/submission_stage5_final.csv"
if os.path.exists(sub_file):
    df_sub = pd.read_csv(sub_file)
    print(f"Submission Shape: {df_sub.shape}")
    print(f"Mean Predicted Probability: {df_sub['Target'].mean():.5f}")
    print(f"5th Percentile: {df_sub['Target'].quantile(0.05):.5f} | 95th Percentile: {df_sub['Target'].quantile(0.95):.5f}")
    print("First 10 Rows:")
    print(df_sub.head(10))
else:
    print(f"{sub_file} not found yet.")

## Stage 7: 20% Immutable Shadow Holdout Verification Benchmark
Partitions an untouched 20% shadow holdout (never seen by feature screening, training, or stacking) to empirically prove that the clean leak-free pipeline outperforms the legacy optimistic pipeline out-of-sample.

In [ ]:
from experiments.run_shadow_holdout_benchmark import run_benchmark

# Execute 20% Shadow Holdout Benchmark
benchmark_passed = run_benchmark()
print(f"Shadow Holdout Verification Success: {benchmark_passed}")
